# SEC 10-K Sentence Labeling Pipeline - Google Colab

This notebook processes SEC 10-K Item extractions into sentence-level datasets ready for manual labeling.

**Workflow:**
1. Setup environment & mount Drive
2. Build sentence table from Item extractions
3. Create balanced labeling sample
4. Download & label in Google Sheets
5. Process labeled data

**Data location:** `/content/drive/MyDrive/sec_10k_project/`

## 📦 Step 1: Environment Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set base path (CUSTOMIZE THIS)
BASE_PATH = '/content/drive/MyDrive/sec_10k_project'

print(f"✓ Drive mounted. Base path: {BASE_PATH}")

In [ ]:
# Create folder structure
import os

folders = [
    'extracted_items',
    'sentence_tables',
    'labeling_samples',
    'labeled_data'
]

for folder in folders:
    path = f'{BASE_PATH}/{folder}'
    os.makedirs(path, exist_ok=True)
    print(f"✓ {path}")

print("\n✓ Folder structure ready")

In [ ]:
# Install dependencies
!pip install -q pandas numpy tqdm pyyaml pyarrow
!pip install -q spacy ftfy contractions chardet opencc-python-reimplemented
!python -m spacy download en_core_web_sm

print("✓ Dependencies installed")

In [ ]:
# Clone cntext repository
!git clone https://github.com/haowenluo/cntext.git /content/cntext

import sys
sys.path.insert(0, '/content/cntext')

# Verify
import cntext as ct
print(f"✓ cntext version: {ct.__version__}")

In [ ]:
# Copy pipeline scripts from tech_adoption_project folder
!cp /content/cntext/tech_adoption_project/build_sentence_table.py /content/
!cp /content/cntext/tech_adoption_project/build_labeling_sample.py /content/
!cp /content/cntext/tech_adoption_project/tech_keywords.yaml /content/

print("✓ Pipeline scripts ready")
print("  - build_sentence_table.py")
print("  - build_labeling_sample.py")
print("  - tech_keywords.yaml")

## 📝 Step 2: Build Sentence Table

**Before running:** Upload your Item extractions to:
`/content/drive/MyDrive/sec_10k_project/extracted_items/`

**Expected format (JSON or CSV):**
```json
{
  "cik": 1234567,
  "accession": "0000000000-20-000001",
  "fiscal_year": 2020,
  "filing_date": "2021-02-15",
  "item": "1",
  "item_text": "Your extracted text..."
}
```

In [ ]:
# List available input files
import os

input_dir = f'{BASE_PATH}/extracted_items'
files = [f for f in os.listdir(input_dir) if f.endswith(('.json', '.csv'))]

print(f"Available input files in {input_dir}:")
for i, f in enumerate(files):
    print(f"  [{i}] {f}")

if not files:
    print("\n⚠️  No files found! Please upload your Item extractions first.")

In [ ]:
# CONFIGURE: Set your input file name
INPUT_FILENAME = 'items_2020.json'  # ← CHANGE THIS

# Run sentence table builder
from build_sentence_table import build_sentence_table

input_file = f'{BASE_PATH}/extracted_items/{INPUT_FILENAME}'
output_file = f'{BASE_PATH}/sentence_tables/sentence_table_{INPUT_FILENAME.split(".")[0]}.csv'

print(f"Input:  {input_file}")
print(f"Output: {output_file}")
print("\nProcessing...\n")

sentence_df = build_sentence_table(
    input_path=input_file,
    output_path=output_file,
    output_format='both'
)

print(f"\n✓ Sentence table created: {len(sentence_df):,} sentences")

In [ ]:
# Verify output
import pandas as pd

df = pd.read_csv(output_file)

print(f"Total sentences: {len(df):,}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSentences by Item:")
print(df['item'].value_counts().sort_index())
print(f"\nSentences by Year:")
print(df['fiscal_year'].value_counts().sort_index())

print(f"\nSample sentences:")
print(df[['cik', 'fiscal_year', 'item', 'sentence_text']].head(5))

## 🎯 Step 3: Build Labeling Sample

Creates a balanced sample of ~2500 sentences:
- 50% from tech keyword hits
- 50% from random background

In [ ]:
# (Optional) Customize tech keywords
import yaml

# View current keywords
with open('/content/tech_keywords.yaml', 'r') as f:
    keywords = yaml.safe_load(f)

print("Current keyword categories:")
for category in keywords['Dictionary'].keys():
    count = len(keywords['Dictionary'][category])
    print(f"  {category}: {count} terms")

# Add custom keywords if needed
# keywords['Dictionary']['custom_category'] = ['term1', 'term2', ...]
# with open('/content/tech_keywords.yaml', 'w') as f:
#     yaml.dump(keywords, f)

In [ ]:
# Run labeling sample builder
from build_labeling_sample import build_labeling_sample
from tqdm import tqdm
tqdm.pandas()

# Configure sample size
SAMPLE_SIZE = 2500  # ← Adjust as needed

input_file = output_file  # Use sentence table from Step 2
label_output = f'{BASE_PATH}/labeling_samples/label_set_{INPUT_FILENAME.split(".")[0]}.csv'

print(f"Input:  {input_file}")
print(f"Output: {label_output}")
print(f"Sample size: {SAMPLE_SIZE}")
print("\nProcessing...\n")

label_df = build_labeling_sample(
    input_path=input_file,
    output_path=label_output,
    keywords_file='/content/tech_keywords.yaml',
    sample_size=SAMPLE_SIZE
)

print(f"\n✓ Labeling sample created: {len(label_df):,} sentences")

In [ ]:
# Review sample quality
print("Source pool distribution:")
print(label_df['source_pool'].value_counts())

print("\nTech hit distribution:")
print(label_df['tech_hit'].value_counts())

print("\n" + "="*80)
print("Sample TECH HITS:")
print("="*80)
tech_samples = label_df[label_df['tech_hit'] == True].head(10)
for idx, row in tech_samples.iterrows():
    print(f"\n[{idx}] {row['sentence_text'][:200]}...")

print("\n" + "="*80)
print("Sample RANDOM:")
print("="*80)
random_samples = label_df[label_df['source_pool'] == 'random'].head(10)
for idx, row in random_samples.iterrows():
    print(f"\n[{idx}] {row['sentence_text'][:200]}...")

## 📥 Step 4: Download for Labeling

**Option A: Open in Google Sheets (Recommended)**
1. Navigate to folder in Drive: `sec_10k_project/labeling_samples/`
2. Right-click CSV → "Open with" → "Google Sheets"
3. Label directly in Sheets (auto-saves!)

**Option B: Download to local**

In [ ]:
# Download to local machine
from google.colab import files

print(f"Downloading: {label_output}")
files.download(label_output)

## ✏️ Labeling Instructions

For each sentence, mark **EXACTLY ONE** column with `1` (leave others blank or `0`):

| Column | Description | Examples |
|--------|-------------|----------|
| `TECH_IMPL` | Technology implementation/usage | "We deployed AI algorithms", "Our cloud infrastructure processes..." |
| `TECH_ADOPT` | Technology adoption/investment | "We invested $50M in R&D", "We acquired a ML company" |
| `TECH_PRODUCT` | Technology product/offering | "Our SaaS platform offers...", "We sell cybersecurity software" |
| `NON_TECH` | Not technology-related | "We operate retail stores", "Revenue increased 10%" |

**Constraint:** If `NON_TECH=1`, all other columns must be `0`

**Tips:**
- `tech_hit` column is just a hint (not always accurate)
- Focus on sentence's **main topic**
- Be consistent across similar sentences

## ✅ Step 5: Validate Labeled Data

**After labeling:** Save as CSV and upload to `sec_10k_project/labeled_data/`

In [ ]:
# Load labeled data
LABELED_FILENAME = 'labeled_set_items_2020.csv'  # ← CHANGE THIS

labeled_file = f'{BASE_PATH}/labeled_data/{LABELED_FILENAME}'
labeled_df = pd.read_csv(labeled_file)

print(f"Loaded {len(labeled_df):,} labeled sentences")

In [ ]:
# Validate labels
label_cols = ['TECH_IMPL', 'TECH_ADOPT', 'TECH_PRODUCT', 'NON_TECH']

print("Label distribution:")
for col in label_cols:
    count = (labeled_df[col] == 1).sum()
    pct = count / len(labeled_df) * 100
    print(f"  {col}: {count:,} ({pct:.1f}%)")

# Quality checks
label_sum = labeled_df[label_cols].sum(axis=1)

unlabeled = (label_sum == 0).sum()
multi_label = (label_sum > 1).sum()
valid = (label_sum == 1).sum()

print(f"\nQuality checks:")
print(f"  ✓ Valid (exactly 1 label): {valid:,} ({valid/len(labeled_df)*100:.1f}%)")
print(f"  ⚠ Unlabeled (0 labels): {unlabeled:,}")
print(f"  ❌ Multi-label (>1 labels): {multi_label:,}")

if multi_label > 0:
    print("\n⚠️  Found multi-label rows (should fix):")
    bad_rows = labeled_df[label_sum > 1][['cik', 'sentence_text'] + label_cols].head(5)
    print(bad_rows)

In [ ]:
# Export clean labeled data
clean_df = labeled_df[label_sum == 1].copy()
clean_output = f'{BASE_PATH}/labeled_data/clean_{LABELED_FILENAME}'

clean_df.to_csv(clean_output, index=False)
print(f"✓ Saved clean labeled data: {len(clean_df):,} sentences")
print(f"  {clean_output}")

## 📊 Next Steps: Train Classifier

Now you have clean labeled data ready for ML training!

**Suggested approaches:**
1. **Simple baseline**: Logistic Regression with TF-IDF
2. **Better performance**: Fine-tune BERT/RoBERTa
3. **Zero-shot**: Use LLM (GPT-4, Claude) via cntext's LLM module

The labeled data is in your Drive at:
`/content/drive/MyDrive/sec_10k_project/labeled_data/`